# Lab 02 — Chapter 19–24: Perplexity và Next-word Prediction

Notebook này thực hiện Experiment 3 (so sánh perplexity) và application dự đoán từ tiếp theo trên cùng corpus.

## 1. Nạp core N-gram và thiết lập thí nghiệm

In [1]:
from pathlib import Path
import json

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_DIRECTORY = next((directory for directory in (WORKING_DIRECTORY, *WORKING_DIRECTORY.parents) if (directory / 'lab2').is_dir() and (directory / 'data').is_dir()), None)
if PROJECT_DIRECTORY is None:
    raise FileNotFoundError('Không tìm thấy thư mục gốc của dự án.')

CORE_NOTEBOOK_PATH = PROJECT_DIRECTORY / 'lab2' / 'Chapter14' / 'experiments.ipynb'
CORE_NOTEBOOK = json.loads(CORE_NOTEBOOK_PATH.read_text(encoding='utf-8'))
CORE_SOURCE = ''.join(CORE_NOTEBOOK['cells'][2]['source'])
exec(CORE_SOURCE)

DATASET_PATH = PROJECT_DIRECTORY / 'data' / 'c4-train.00000-of-01024-30K.json' / 'c4-train.00000-of-01024-30K.json'
DOCUMENT_LIMIT = 10_000
TRAIN_SHARE = 0.80
VALIDATION_SHARE = 0.10
NGRAM_ORDERS = (1, 2, 3)
MODEL_NAMES = {1: 'Unigram', 2: 'Bigram', 3: 'Trigram'}
USE_LAPLACE_SMOOTHING = True
PREDICTION_LIMIT = 3
ACTUAL_NEXT_WORDS = {
    'the cat': 'sat',
    'natural language': 'processing',
    'machine learning': 'models',
    'data science': 'is',
    'artificial intelligence': 'is',
}

def split_corpus(sentences):
    """Chia tuần tự corpus thành train, validation và test theo tỉ lệ cấu hình."""
    sentence_count = len(sentences)
    training_end = max(1, int(sentence_count * TRAIN_SHARE))
    validation_end = min(sentence_count - 1, max(training_end + 1, int(sentence_count * (TRAIN_SHARE + VALIDATION_SHARE))))
    return sentences[:training_end], sentences[training_end:validation_end], sentences[validation_end:]


def predict_next(model, context, limit):
    """Trả về các từ có xác suất điều kiện cao nhất sau context."""
    context_tokens = tokenize(context)
    candidates = ((word, model.probability(context_tokens, word)) for word in model.vocabulary if word != UNKNOWN_TOKEN)
    return sorted(candidates, key=lambda item: item[1], reverse=True)[:limit]


## 2. Chuẩn bị train, validation và test corpus

In [2]:
document_count, sentences = load_sentences(DATASET_PATH, DOCUMENT_LIMIT)
training_corpus, validation_corpus, test_corpus = split_corpus(sentences)
evaluation_sets = (
    ('Train', training_corpus),
    ('Validation', validation_corpus),
    ('Test', test_corpus),
)
display({
    'documents': document_count,
    'train_sentences': len(training_corpus),
    'validation_sentences': len(validation_corpus),
    'test_sentences': len(test_corpus),
})


{'documents': 10000,
 'train_sentences': 164280,
 'validation_sentences': 20535,
 'test_sentences': 20535}

## 3. Chapter 19 — Experiment 3: So sánh perplexity

In [3]:
models = {}
perplexity_results = []

for order in NGRAM_ORDERS:
    model = NGramLanguageModel(order=order, smoothing=USE_LAPLACE_SMOOTHING).fit(training_corpus)
    models[order] = model
    result_row = {'Model': MODEL_NAMES[order]}
    for split_name, evaluation_corpus in evaluation_sets:
        result_row[split_name] = model.perplexity(evaluation_corpus)
    perplexity_results.append(result_row)

display(perplexity_results)


[{'Model': 'Unigram',
  'Train': 1857.6346368763238,
  'Validation': 1992.4122081600128,
  'Test': 2019.1579861319642},
 {'Model': 'Bigram',
  'Train': 5156.916949707187,
  'Validation': 7237.766258961776,
  'Test': 7308.537407939962},
 {'Model': 'Trigram',
  'Train': 19055.04545669437,
  'Validation': 33451.40039488381,
  'Test': 33241.2465181789}]

### Giải thích kết quả Experiment 3

- Training perplexity thường giảm khi chuyển từ unigram sang bigram/trigram vì context dài hơn giúp mô hình ghi nhận nhiều mẫu đã thấy trong train.
- Validation/test perplexity không nhất thiết giảm: context dài làm số N-gram hiếm hoặc chưa thấy tăng, nên mô hình có thể khái quát hóa kém hơn.
- Dấu hiệu overfitting là PPL train tiếp tục giảm, trong khi PPL validation/test tăng hoặc không cải thiện.
- Corpus lớn hơn làm count tin cậy hơn và giảm sparsity; nhờ đó bigram/trigram thường hưởng lợi nhiều hơn unigram. Tuy nhiên, dữ liệu phải cùng domain và được tiền xử lý nhất quán.

## 4. Chapter 20–24 — Application: Next-word prediction

In [4]:
best_order = min(perplexity_results, key=lambda row: row['Validation'])['Model']
selected_order = next(order for order, name in MODEL_NAMES.items() if name == best_order)
selected_model = models[selected_order]
next_word_results = []

for context, actual_next_word in ACTUAL_NEXT_WORDS.items():
    predictions = predict_next(selected_model, context, PREDICTION_LIMIT)
    predicted_word, probability = predictions[0]
    next_word_results.append({
        'Context': context,
        'Prediction': predicted_word,
        'Probability': probability,
        'Actual next word': actual_next_word,
        'Top 3 predictions': predictions,
    })

display({
    'selected_model': MODEL_NAMES[selected_order],
    'selection_metric': 'lowest validation perplexity',
})
display(next_word_results)


{'selected_model': 'Unigram',
 'selection_metric': 'lowest validation perplexity'}

[{'Context': 'the cat',
  'Prediction': 'the',
  'Probability': 0.050332617529195454,
  'Actual next word': 'sat',
  'Top 3 predictions': [('the', 0.050332617529195454),
   ('and', 0.02914495016901963),
   ('to', 0.027897106617658235)]},
 {'Context': 'natural language',
  'Prediction': 'the',
  'Probability': 0.050332617529195454,
  'Actual next word': 'processing',
  'Top 3 predictions': [('the', 0.050332617529195454),
   ('and', 0.02914495016901963),
   ('to', 0.027897106617658235)]},
 {'Context': 'machine learning',
  'Prediction': 'the',
  'Probability': 0.050332617529195454,
  'Actual next word': 'models',
  'Top 3 predictions': [('the', 0.050332617529195454),
   ('and', 0.02914495016901963),
   ('to', 0.027897106617658235)]},
 {'Context': 'data science',
  'Prediction': 'the',
  'Probability': 0.050332617529195454,
  'Actual next word': 'is',
  'Top 3 predictions': [('the', 0.050332617529195454),
   ('and', 0.02914495016901963),
   ('to', 0.027897106617658235)]},
 {'Context': 'ar

### Ghi nhận prediction

Bảng trên ghi lại đủ 5 contexts cùng prediction, probability và actual next word. Prediction được lấy từ mô hình có validation perplexity thấp nhất, thay vì chọn theo test set. Nếu prediction khác actual word, cần xem xét context có quá ngắn không, N-gram có chưa từng xuất hiện không, vocabulary có giới hạn không, corpus có khác domain không và smoothing có phân bổ xác suất quá đều không.